# XGBoost–SARIMA inference
Loads the experiment artifact, predicts raw `test.csv`, applies the learned holiday-aware blend, and writes a Kaggle-ready submission.

In [ ]:
%pip install -q "xgboost>=3,<4" "statsmodels>=0.14,<1" "joblib>=1.4,<2"

In [ ]:
from pathlib import Path
import warnings, joblib, numpy as np, pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
warnings.filterwarnings('ignore')
DATA_DIR=Path('/content/drive/MyDrive/walmart_competition_data') if Path('/content').exists() else Path('../../../data')
MODEL_PATH=Path('/content/drive/MyDrive/walmart_models/xgboost_sarima_pipeline.joblib') if Path('/content').exists() else Path('artifacts/xgboost_sarima_pipeline.joblib')
bundle=joblib.load(MODEL_PATH); raw=pd.read_csv(DATA_DIR/'test.csv',parse_dates=['Date'])
test=raw.merge(bundle['features_table'],on=['Store','Date','IsHoliday'],how='left').merge(bundle['stores_table'],on='Store',how='left')

In [ ]:
def make_x(df,history):
    z=df.copy(); z['Type']=z.Type.map({'A':0,'B':1,'C':2}); z['year']=z.Date.dt.year; z['month']=z.Date.dt.month
    z['week']=z.Date.dt.isocalendar().week.astype(int); z['week_sin']=np.sin(2*np.pi*z.week/52); z['week_cos']=np.cos(2*np.pi*z.week/52)
    lag=history.copy(); lag.Date=lag.Date+pd.Timedelta(weeks=52); lag=lag.rename(columns={'Weekly_Sales':'lag_52'})
    z=z.merge(lag,on=['Store','Dept','Date'],how='left')
    return z[bundle['feature_columns']].astype(float).fillna(-999)
px=bundle['xgb'].predict(make_x(test,bundle['history'])); ps=np.full(len(test),np.nan)
for key,g in test.groupby(['Store','Dept'],sort=False):
    y=bundle['history'].query('Store==@key[0] and Dept==@key[1]').sort_values('Date').Weekly_Sales
    if len(y)<80: continue
    try: ps[test.index.get_indexer(g.index)]=SARIMAX(y,order=bundle['order'],seasonal_order=bundle['seasonal_order'],enforce_stationarity=False,enforce_invertibility=False).fit(disp=False,maxiter=60).forecast(len(g))
    except Exception: pass
ps=np.where(np.isfinite(ps),ps,px); weight=np.where(test.IsHoliday,bundle['holiday_xgb_weight'],bundle['normal_xgb_weight'])
prediction=weight*px+(1-weight)*ps

In [ ]:
submission=pd.DataFrame({'Id':raw.Store.astype(str)+'_'+raw.Dept.astype(str)+'_'+raw.Date.dt.strftime('%Y-%m-%d'),'Weekly_Sales':prediction})
assert len(submission)==len(raw) and submission.Weekly_Sales.notna().all()
submission.to_csv('submission_xgboost_sarima.csv',index=False)
submission.head(), submission.Weekly_Sales.describe()